** Binary Connected Components Implementation ** 

In [64]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import label
from collections import defaultdict
import json

F = np.loadtxt(
    "f_nearest_neighbors_lp.csv",
    delimiter=","
)

nRows = 42
nCols = 44

print(F.shape)

(1848, 20)


In [65]:
def get_matrix_u(F, tree):
    U = F.copy()

    for parent, children in tree.items():
        for child in children:
            U[:, parent] -= F[:, child]

    return U

In [66]:
frontend_tree = {
    1: [2, 3, 4, 5, 6, 7, 11, 12, 13],
    2: [9, 10],
    3: [14],
    4: [15, 16],
    5: [8],
    6: [],
    7: [],
    8: [17, 18, 19],
    9: [],
    10: [],
    11: [],
    12: [],
    13: [],
    14: [],
    15: [],
    16: [],
    17: [],
    18: [],
    19: [20],
    20: []
}

frontend_tree_zero = {
    parent - 1: [child - 1 for child in children]
    for parent, children in frontend_tree.items()
}

In [67]:
U = get_matrix_u(F, frontend_tree_zero)

In [68]:

#generates binary connected components, 4 connectivity
def binary_connected_components(grid, threshold=0.1):
    binary = grid >= threshold

    # 4 connectivity means cells r connected if they share an edge (not including corners)
    structure = np.array([
        [0, 1, 0],
        [1, 1, 1],
        [0, 1, 0]
    ])

    labeled_grid, num_components = label(
        binary,
        structure=structure
    )

    return labeled_grid, num_components

In [69]:
def extract_boundary_edges(labeled_grid, component_id):
    rows, cols = labeled_grid.shape
    edges = []

    for r in range(rows):
        for c in range(cols):

            if labeled_grid[r, c] != component_id:
                continue

            # Top edge
            if r == 0 or labeled_grid[r-1, c] != component_id:
                edges.append(((c, r), (c+1, r)))

            # Bottom edge
            if r == rows-1 or labeled_grid[r+1, c] != component_id:
                edges.append(((c, r+1), (c+1, r+1)))

            # Left edge
            if c == 0 or labeled_grid[r, c-1] != component_id:
                edges.append(((c, r), (c, r+1)))

            # Right edge
            if c == cols-1 or labeled_grid[r, c+1] != component_id:
                edges.append(((c+1, r), (c+1, r+1)))

    return edges

In [70]:
### Boundary stitching 
def stitch_boundary_edges(edges):
    adjacency = defaultdict(list)

    # Map each vertex to its neighboring vertices
    for start, end in edges:
        adjacency[start].append(end)
        adjacency[end].append(start)

    unused = {
        frozenset((start, end))
        for start, end in edges
    }

    paths = []

    while unused:
        start, current = tuple(next(iter(unused)))
        path = [start, current]

        unused.remove(frozenset((start, current)))

        # Follow edges until we return to the start
        while current != start:
            for neighbor in adjacency[current]:
                edge = frozenset((current, neighbor))

                if edge in unused:
                    unused.remove(edge)
                    path.append(neighbor)
                    current = neighbor
                    break
            else:
                raise ValueError("Boundary path could not be closed")

        paths.append(path)

    return paths

In [71]:
boundaries = []

for component_id in range(1, num_components + 1):

    edges = extract_boundary_edges(
        labeled_grid,
        component_id
    )

    paths = stitch_boundary_edges(edges)

    boundaries.extend(paths)

boundaries

[[(11, 35),
  (10, 35),
  (10, 36),
  (9, 36),
  (9, 37),
  (10, 37),
  (10, 38),
  (11, 38),
  (11, 37),
  (11, 36),
  (11, 35)],
 [(15, 38),
  (16, 38),
  (17, 38),
  (17, 39),
  (17, 40),
  (17, 41),
  (16, 41),
  (15, 41),
  (15, 40),
  (14, 40),
  (14, 39),
  (15, 39),
  (15, 38)]]

In [72]:
threshold = 0.1
U = get_matrix_u(F, trees[0])

all_boundaries = {}

for clone_idx in range(U.shape[1]):

    grid = U[:, clone_idx].reshape(nRows, nCols)

    labeled_grid, num_components = (
        binary_connected_components(grid, threshold)
    )

    boundaries = []

    for component_id in range(1, num_components + 1):

        edges = extract_boundary_edges(
            labeled_grid,
            component_id
        )

        paths = stitch_boundary_edges(edges)

        boundaries.extend(paths)

    # Adjust coordinates for frontend grid padding
    all_boundaries[str(clone_idx)] = [
        [[x + 0.5, y + 0.5] for x, y in path]
        for path in boundaries
    ]

with open("cc_boundaries.json", "w") as file:
    json.dump(all_boundaries, file)